Main variables

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime

base_path = "s3://.../events/"
sf_target_table = "EVENTS_LANDING"
sf_reject_table = "REJECTED_EVENTS"
sf_dates_table = "LOADED_DATES"

sfOptions = {
  "sfURL": dbutils.secrets.get("snowflake_retail_scope", "snowflake_host"),
  "sfUser": dbutils.secrets.get("snowflake_retail_scope", "snowflake_user"),
  "sfPassword": dbutils.secrets.get("snowflake_retail_scope", "snowflake_password"),
  "sfDatabase": dbutils.secrets.get("snowflake_retail_scope", "snowflake_database"),
  "sfSchema": dbutils.secrets.get("snowflake_retail_scope", "snowflake_schema"),
  "sfWarehouse": dbutils.secrets.get("snowflake_retail_scope", "snowflake_warehouse"),
  "sfRole": dbutils.secrets.get("snowflake_retail_scope", "snowflake_role"),
  "tempDir": "s3://rigp-retail-dev-bronze/sf_temp/"
}

Missing dates

In [ ]:
date_folders = dbutils.fs.ls(base_path) 
    # [dbutils] Databricks Utilities. 
    # [fs] File System (para trabajar con archivos o rutas). 
    # [ls] List (para listar lo que hay dentro).
    # Es decir, va al path de la carpeta y lista todo lo que hay adentro, es decir los "event_date=<fecha>/"

available_dates = []
for it in date_folders:
    name = it.name.rstrip("/")
    if name.startswith("event_date="):
        date_str = name.replace("event_date=", "")
        try:
            d = datetime.strptime(date_str, "%Y-%m-%d").date()
            available_dates.append(d)
        except:
            pass

available_dates_set = set(available_dates) # set() crea un conjunto, que luego es facil para ver que fechas faltan: set1 - set2

loaded_dates_df = (
    spark.read
         .format("snowflake")
         .options(**sfOptions)
         .option("query", f"SELECT DISTINCT EVENT_DATE FROM {sf_dates_table}")
         .load()
)

loaded_dates_set = set()
for it in loaded_dates_df.collect(): # collect() en spark trae todos los registros de un df como una lista de filas
    d = it["EVENT_DATE"]
    loaded_dates_set.add(d.date() if hasattr(d, "date") else d) # Checkea que sea tipo date, sino si es datetime lo pasa.

missing_dates_set = available_dates_set - loaded_dates_set

Data transformation and write

In [ ]:
from pyspark.sql import DataFrame

INDEX_COL = "__index_level_0__"
BASE_COLS = [
    "event_time", "event_date", "load_date",
    "price", "product_id", "user_id", "category_id",
    "event_type", "category_code", "brand", "user_session"
]
REJECT_EXTRA_COLS = ["reject_reason", "rejected_at"]

def standarize_events(df_events: DataFrame) -> DataFrame:
    std_events = (
        df_events
        # ------- Tiempo -------
        .withColumn("event_time", F.col("event_time").cast("timestamp"))
        .withColumn("event_date", F.to_date("event_time"))
        # ------ Numeros -------
        .withColumn("price", F.col("price").cast("decimal(10,2)"))
        .withColumn("product_id", F.col("product_id").cast("long"))
        .withColumn("user_id", F.col("user_id").cast("long"))
        .withColumn("category_id", F.col("category_id").cast("long"))
        # ------ Strings -------
        .withColumn("event_type", F.trim(F.col("event_type")).cast("string"))
        .withColumn("category_code", F.trim(F.col("category_code")).cast("string"))
        .withColumn("brand", F.trim(F.col("brand")).cast("string"))
        .withColumn("user_session", F.trim(F.col("user_session")).cast("string"))
    )
    if INDEX_COL in std_events.columns:
        std_events = std_events.drop("__index_level_0__")
    return std_events

def write_to_snowflake(df_to_load: DataFrame, sf_table: str, sfOptions: dict, mode = "append") -> None:
    (
        df_to_load.write
        .format("snowflake")
        .options(**sfOptions)
        .option("dbtable", sf_table)
        .mode(mode)
        .save()
    )

Filtrar las fechas para simular un batch semanal

In [ ]:
dates_to_load = sorted(missing_dates_set)[:7]

Transformar y subir los datos

In [ ]:
run_timestamp = F.current_timestamp()
paths = [f"{base_path}event_date={d}" for d in dates_to_load]

if len(paths) == 0:
    print("No hay fechas nuevas para cargar.")
else:
    df_raw = spark.read.parquet(*paths)
    df_std = standarize_events(df_raw)

    df_clean = (
        df_std
        .where(F.col("user_session").isNotNull() & (F.col("user_session") != "")) # De paso tambien agrego que no sea un espacio vacio
        .select([c for c in BASE_COLS if c != "load_date"])
    )
    write_to_snowflake(df_clean, sf_target_table, sfOptions)

    df_rejected = (
        df_std
        .where(F.col("user_session").isNull() | (F.col("user_session") == ""))
        .withColumn("reject_reason", F.when(F.col("user_session").isNull(), F.lit("USER_SESSION_IS_NULL")).otherwise(F.lit("USER_SESSION_IS_EMPTY")))
        .withColumn("rejected_at", run_timestamp)
        .withColumn("load_date", run_timestamp)
        .select(*(BASE_COLS + REJECT_EXTRA_COLS))
    )
    write_to_snowflake(df_rejected, sf_reject_table, sfOptions)

    df_clean_grouped = (
        df_clean
        .groupBy("event_date")
        .agg(
            run_timestamp.alias("loaded_date"),
            F.count("*").alias("rows_loaded")
        )
    )

    df_rejected_grouped = (
        df_rejected
        .groupBy("event_date")
        .agg(F.count("*").alias("rows_rejected"))
    )

    df_dates = (
        df_clean_grouped
        .join(df_rejected_grouped, on="event_date", how="left")
        .fillna({"rows_rejected": 0})
        .select("event_date", "loaded_date", "rows_loaded", "rows_rejected")
    )

    write_to_snowflake(df_dates, sf_dates_table, sfOptions)